Gauntlet #3: The Centering Problem

## 📘 Day 3, Gauntlet #3: The Centering Problem

### What This Gauntlet Tests
This challenge combines **broadcasting**, **axis-wise reduction**, and **`keepdims=True`** into a realistic data normalization pipeline. The data has shape `(100, 50, 20)`:
- **100** = samples (e.g., 100 data points / images)
- **50** = rows (e.g., feature rows)
- **20** = columns (e.g., feature columns)

Each of the 3 steps centers or scales the data along a different axis combination, and you must use broadcasting correctly to subtract/divide without changing the shape.

---

### Key Tool: `keepdims=True`
Without `keepdims`, the reduced axis disappears and broadcasting fails:
```python
data = np.random.rand(100, 50, 20)

# ❌ Wrong — keepdims not used
mean = data.mean(axis=(1, 2))         # shape (100,) — can't subtract from (100,50,20)
data - mean                           # ERROR

# ✓ Correct — shape preserved as (100, 1, 1) → broadcasts to (100, 50, 20)
mean = data.mean(axis=(1, 2), keepdims=True)
data - mean                           # (100,50,20) - (100,1,1) → (100,50,20)
```

---

### Step 1: Center Each Sample (subtract per-sample mean)
**Goal:** For each of the 100 samples, subtract that sample's overall mean across its entire 50×20 feature map.

```
Reduce over: axis=(1, 2)   ← average over all rows AND all columns per sample
keepdims result shape: (100, 1, 1)
Broadcasts against:    (100, 50, 20)  ✓
```

```python
data_mean = data.mean(axis=(1, 2), keepdims=True)   # shape (100, 1, 1)
data_centered = data - data_mean                     # shape (100, 50, 20)
```
After this step: every sample independently has zero mean.

---

### Step 2: Subtract Row-wise Global Mean
**Goal:** For each of the 50 rows, compute one mean value averaged across all 100 samples AND all 20 columns. Subtract it from every element in that row.

```
Reduce over: axis=(0, 2)   ← average over samples AND columns, keeping row axis
keepdims result shape: (1, 50, 1)
Broadcasts against:    (100, 50, 20)  ✓
```

```python
row_mean = data.mean(axis=(0, 2), keepdims=True)     # shape (1, 50, 1)
data_row_centered = data_centered - row_mean          # shape (100, 50, 20)
```
After this step: each row has its global average removed.

---

### Step 3: Normalize Columns by Standard Deviation
**Goal:** For each of the 20 columns, compute one standard deviation averaged across all 100 samples AND all 50 rows. Divide every element in that column by it.

```
Reduce over: axis=(0, 1)   ← std over samples AND rows, keeping column axis
keepdims result shape: (1, 1, 20)
Broadcasts against:    (100, 50, 20)  ✓
```

```python
col_std = data.std(axis=(0, 1), keepdims=True)       # shape (1, 1, 20)
data_normalized = data_row_centered / col_std         # shape (100, 50, 20)
```
After this step: each column has unit standard deviation globally.

---

### The Full Pipeline at a Glance

| Step | Axis reduced | keepdims shape | What it removes |
|------|-------------|----------------|-----------------|
| 1 | `(1, 2)` | `(100, 1, 1)` | Per-sample offset |
| 2 | `(0, 2)` | `(1, 50, 1)` | Per-row global bias |
| 3 | `(0, 1)` | `(1, 1, 20)` | Per-column scale |

> 💡 **Pattern to remember:** The axes you pass to `keepdims=True` become the `1`s in the output shape. The axes you **don't** reduce stay as-is. The 1s then broadcast freely against the original shape.

In [1]:
import numpy as np
np.random.seed(42)  # for reproducibility
data = np.random.rand(100, 50, 20)  # shape (100, 50, 20)

In [13]:
#Center each sample individually
#Subtract the mean of each sample's entire 50×20 feature map so that each sample
#has zero mean.
#Result shape must be (100, 50, 20)
data_mean = data.mean(axis=(1, 2), keepdims=True)
data_centered = data - data_mean
print(data_centered.shape)

(100, 50, 20)


In [14]:
#Subtract row‑wise global mean
#After step 1, compute the global mean of each row across all 
#samples and columns. That is, for each of the 50 rows, 
#compute the mean over all 100 samples and 20 columns. 
#Then subtract that row‑specific mean from every element in 
#that row across all samples and columns.
#Result shape: (100, 50, 20).
row_mean = data.mean(axis=(0, 2), keepdims=True)
data_row_centered = data_centered - row_mean 
print(data_row_centered.shape)

(100, 50, 20)


In [16]:
#Normalize columns by standard deviation
#After step 2, compute the standard deviation of each column across 
#all samples and rows (so one value per column). 
#Divide each column by its standard deviation (broadcasting).
#Result shape: (100, 50, 20).
col_std = data.std(axis=(0, 1), keepdims=True)
data_col_centered = data_row_centered / col_std
print(data_col_centered.shape)

(100, 50, 20)
